In [7]:
import pandas as pd

from user import User
from pyBKT.models import *
from pyBKT.models import Model
from matplotlib import pyplot as plt
from bkt_preprocessing import extract_mc


model = Model(seed=52, num_fits=10)
model.load(loc='without_forget.pkl')

ratings = pd.read_csv('../data/ml-1m/ratings.csv')
_, df = extract_mc(num_topics=20, rating_data=ratings, ratio=0.2, flag=False)
params = model.params().reset_index()

In [8]:
def extract_mc_by_ratio(row, ratio):
    idx = row[row >= ratio].index
    mcs = [i.split()[1] for i in idx]

    return mcs

In [9]:
# 추천 후보 리스트 생성
movies = pd.read_csv('../data/crawled_data/plots_crawled.csv')
lda = pd.read_csv('../LDA/result/result_20/20240626_Topic=20_TopicDist.csv')

lda['mc'] = lda.apply(lambda x: extract_mc_by_ratio(row=x, ratio=0.2), axis=1)

movies = movies[(movies['plots'] != 'no results') & (movies['plots'] != 'no synopsis') & (~movies['plots'].isna())]
lda['movieId'] = movies['movieId']

assert(movies.shape[0] == lda.shape[0])

rec_cand = ratings.merge(right=lda.loc[:, ['movieId', 'mc']], how='inner', on='movieId').drop_duplicates(subset='movieId').sort_values(by='movieId').reset_index(drop=True)

# 영화 제목 병합하기
candidates = ratings['movieId'].unique()

title = pd.read_csv('../data/crawled_data/plots_crawled.csv')
title = movies[movies['movieId'].isin(candidates)].reset_index(drop=True)
title = movies[['movieId', 'title']]

rec_cand = rec_cand[['movieId', 'mc']]
rec_cand = rec_cand.merge(right=title, how='inner', on='movieId')

rec_cand_temp = rec_cand.explode(column='mc', ignore_index=True)

In [10]:
def compute_prob(movie_id, mc, pf_state):
    prior = pf_state[mc]
    slip = params[
        (params['skill'] == mc) & 
        (params['param'] == 'slips') & 
        (params['class'] == str(movie_id))
        ]['value'].values[0]
    guess = params[
        (params['skill'] == mc) & 
        (params['param'] == 'guesses') & 
        (params['class'] == str(movie_id))
    ]['value'].values[0]

    return prior * (1 - slip) + (1 - prior) * guess


# history
user = User(user_id=1679, params=params)
interactions = df[df['userId'] == 1679]
user.update_state(movie_ids=interactions['movieId'].values, mcs=interactions['mc'].values, responses=interactions['rating'].values)


before = 20
after = 120

# before
rec_cand_temp[f'pred_prob_t={before}'] = rec_cand_temp.apply(lambda x: compute_prob(movie_id=x['movieId'], mc=x['mc'], pf_state=user.get_history(t=before)), axis=1)

# after
rec_cand_temp[f'pred_prob_t={after}'] = rec_cand_temp.apply(lambda x: compute_prob(movie_id=x['movieId'], mc=x['mc'], pf_state=user.get_history(t=after)), axis=1)

In [11]:
user.get_history()[120]['11']

0.5961622903040307

In [12]:
pred_prob_before = rec_cand.merge(right=rec_cand_temp.groupby(by='title')[f'pred_prob_t={before}'].mean(), how='inner', on='title')[f'pred_prob_t={before}']
pred_prob_after = rec_cand.merge(right=rec_cand_temp.groupby(by='title')[f'pred_prob_t={after}'].mean(), how='inner', on='title')[f'pred_prob_t={after}']

rec_cand[f'pred_prob_t={before}'] = pred_prob_before
rec_cand[f'pred_prob_t={after}'] = pred_prob_after

In [13]:
# recommendation list; before
rec_cand.sort_values(by=f'pred_prob_t={before}', ascending=False).head(100)

,movieId,mc,title,pred_prob_t=20,pred_prob_t=120
724,787,"[2, 14]","Gate of Heavenly Peace, The (1995)",1.00000,1.00000
1624,1842,[14],Illtown (1996),1.00000,1.00000
728,792,[13],"Hungarian Fairy Tale, A (1987)",1.00000,1.00000
132,139,[7],Target (1995),1.00000,1.00000
3300,3607,[1],One Little Indian (1973),1.00000,1.00000
...,...,...,...,...,...
971,1066,"[13, 18]",Shall We Dance? (1937),0.86957,0.85262
2322,2571,[18],"Matrix, The (1999)",0.86763,0.83863
1232,1361,[18],Paradise Lost: The Child Murders at Robin Hood...,0.86700,0.83314
1145,1267,"[5, 18]","Manchurian Candidate, The (1962)",0.86696,0.85931


In [14]:
# recommendation list at; after
rec_cand.sort_values(by=f'pred_prob_t={after}', ascending=False).head(10)

,movieId,mc,title,pred_prob_t=20,pred_prob_t=120
557,584,[13],I Don't Want to Talk About It (De eso no se ha...,1.00000,1.00000
654,701,"[6, 16]",Daens (1992),1.00000,1.00000
3015,3305,[13],Bluebeard (1944),1.00000,1.00000
127,134,"[10, 13]",Sonic Outlaws (1995),1.00000,1.00000
594,624,[2],Condition Red (1995),1.00000,1.00000
3060,3353,[13],"Closer You Get, The (2000)",1.00000,1.00000
3087,3382,"[10, 13]",Song of Freedom (1936),1.00000,1.00000
1033,1139,"[1, 14]",Everything Relative (1996),1.00000,1.00000
382,398,"[7, 14]",Frank and Ollie (1995),1.00000,1.00000
3218,3517,[13],"Bells, The (1926)",1.00000,1.00000


In [15]:
rec_cand['diff'] = rec_cand['pred_prob_t=120'] - rec_cand['pred_prob_t=20']
rec_cand.sort_values(by='diff', ascending=False).head(10)

,movieId,mc,title,pred_prob_t=20,pred_prob_t=120,diff
714,776,[11],Babyfever (1994),0.19387,0.63438,0.44051
1479,1651,[11],Telling Lies in America (1997),0.04073,0.25008,0.20935
1985,2211,"[6, 11]",Secret Agent (1936),0.33169,0.49131,0.15962
1304,1444,[18],Guantanamera (1994),0.11925,0.25497,0.13572
1004,1102,[18],American Strays (1996),0.04084,0.15630,0.11546
510,533,[11],"Shadow, The (1994)",0.05799,0.15918,0.10118
2633,2896,"[7, 11]",Alvarez Kelly (1966),0.32514,0.41539,0.09026
3143,3441,"[6, 11]",Red Dawn (1984),0.30050,0.38844,0.08794
2424,2679,"[8, 18]",Finding North (1999),0.12819,0.20814,0.07995
218,227,[11],Drop Zone (1994),0.06743,0.14389,0.07646
